[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/halla-ai/deepnlp-2026/blob/main/notebooks/week-08.ipynb)

# 8주차 실습: 중간고사 대비 자가 점검과 PEFT 심화

**목표.** 1~7주차의 핵심 계산을 **먼저 손으로 예측하고, 셀을 실행해 확인**한다. 새 주제는 없다.

- 문항마다 문제를 읽고, 종이나 아래 답 칸에 예측을 먼저 적는다
- 그다음 셀을 실행해 예측과 비교한다. 틀렸으면 문항 끝에 적힌 주차의 강의 노트로 돌아간다
- 3주차 PEFT 문항은 실습 모델 Qwen3-0.6B-Base에 LoRA, DoRA를 실제로 붙여 학습 파라미터 수를 세어 본다(PEFT 심화)

이 노트북에는 바꿀 곳(TODO)이 없다. 처음부터 끝까지 그대로 실행하면 된다. 문항은 시험 유형을 연습하려고 만든 예시이며 실제 시험 문항이 아니다.

## 0. 준비

아래 셀을 실행해 필요한 라이브러리를 설치한다. GPU는 필요 없다.

In [ ]:
# 필요한 것 설치 (Colab에서 한 번만)
!pip -q install -U transformers peft
!pip -q install torch

In [ ]:
import math, torch
torch.manual_seed(0)
def check(name, got, expected, tol=1e-3):
    ok = abs(got - expected) <= tol * max(1, abs(expected))
    print(f"[{'맞음' if ok else '다름'}] {name}: 계산 {got:,.4f} / 기대 {expected:,.4f}")

## 1. 어텐션 한 칸 손계산 (1주차)

토큰 3개, 차원 2인 Query, Key, Value가 있다. 첫 토큰의 Query로 어텐션을 계산한다.

- q = (1, 0), K = [(1, 0), (0, 1), (1, 1)], V = [(1, 0), (0, 1), (1, 1)]
- 점수 = q · k / √2 (차원 2의 제곱근으로 나눈다)

**예측해 보기.** (1) 세 점수는? (2) 소프트맥스 뒤 첫 칸의 가중치는? (3) 출력 벡터는? (4) 토큰이 1,000개면 점수표는 몇 칸인가?

In [ ]:
q = torch.tensor([1.0, 0.0])
K = torch.tensor([[1.0, 0.0], [0.0, 1.0], [1.0, 1.0]])
V = K.clone()
scores = K @ q / math.sqrt(2)
w = torch.softmax(scores, dim=0)
out = w @ V
print("점수:", [round(x, 4) for x in scores.tolist()])
print("가중치:", [round(x, 4) for x in w.tolist()])
print("출력:", [round(x, 4) for x in out.tolist()])
check("첫 칸 가중치", w[0].item(), math.exp(1 / math.sqrt(2)) / (2 * math.exp(1 / math.sqrt(2)) + 1))
print(f"토큰 1,000개의 점수표: {1000 * 1000:,}칸 (n x n)")

## 2. 학습 루프 한 걸음 (2주차)

모델 ŷ = w · x, 손실 L = (ŷ - y)². x = 2, y = 10, 처음 w = 3, 학습률 0.1이다.

**예측해 보기.** (1) 처음 손실은? (2) 기울기 dL/dw는? (3) 한 걸음 뒤 w와 손실은?

In [ ]:
w = torch.tensor(3.0, requires_grad=True)
x, y, lr = 2.0, 10.0, 0.1
loss = (w * x - y) ** 2
loss.backward()
print(f"처음 손실 {loss.item():.2f}, 기울기 {w.grad.item():.2f}")
with torch.no_grad():
    w_new = w - lr * w.grad
new_loss = (w_new * x - y) ** 2
print(f"한 걸음 뒤 w = {w_new.item():.2f}, 손실 {new_loss.item():.2f}")
check("기울기 2 x (wx - y) x x", w.grad.item(), 2 * (3 * 2 - 10) * 2)

## 3. PEFT 심화 - Qwen3-0.6B에 LoRA를 붙이면 (3주차)

3주차는 KoELECTRA(1.1억 개)에 LoRA를 붙였다. 같은 계산을 4~7주차 실습 모델 Qwen3-0.6B-Base에 적용한다. 설정만 내려받아 무작위 가중치로 모델을 만든다(가중치 파일은 내려받지 않는다). 파라미터 **개수**를 세는 데는 가중치 값이 필요 없기 때문이다.

한 층의 선형 층 모양(입력 -> 출력)은 다음과 같다. 이 층이 28번 반복된다.

| 모듈 | 모양 | 모듈 | 모양 |
|---|---|---|---|
| q_proj | 1024 -> 2048 | gate_proj | 1024 -> 3072 |
| k_proj | 1024 -> 1024 | up_proj | 1024 -> 3072 |
| v_proj | 1024 -> 1024 | down_proj | 3072 -> 1024 |
| o_proj | 2048 -> 1024 | | |

**예측해 보기.** 랭크 r의 LoRA는 입력 a, 출력 b인 층에 r x (a + b)개를 붙인다.
(1) r = 8로 q_proj, v_proj에만 붙이면 28층 전체 몇 개인가? (2) 일곱 모듈 모두에 붙이면? (3) DoRA는 붙인 층마다 크기 벡터(출력 차원 길이)를 더한다. q_proj, v_proj에 r = 8 DoRA면 LoRA보다 몇 개 많은가?

In [ ]:
from transformers import AutoConfig, AutoModelForCausalLM
from peft import LoraConfig, get_peft_model

cfg = AutoConfig.from_pretrained("Qwen/Qwen3-0.6B-Base")

def count(targets, r=8, dora=False):
    base = AutoModelForCausalLM.from_config(cfg, dtype=torch.bfloat16)  # 무작위 가중치, 개수만 센다
    total = sum(p.numel() for p in base.parameters())
    pm = get_peft_model(base, LoraConfig(r=r, lora_alpha=2 * r, target_modules=targets, use_dora=dora))
    trainable = sum(p.numel() for p in pm.parameters() if p.requires_grad)
    return trainable, total

L = cfg.num_hidden_layers
shape = {"q_proj": (1024, 2048), "k_proj": (1024, 1024), "v_proj": (1024, 1024), "o_proj": (2048, 1024),
         "gate_proj": (1024, 3072), "up_proj": (1024, 3072), "down_proj": (3072, 1024)}
def lora_formula(targets, r=8):
    return L * sum(r * (shape[t][0] + shape[t][1]) for t in targets)

qv = ["q_proj", "v_proj"]
all7 = list(shape)
t_qv, total = count(qv)
t_all, _ = count(all7)
t_dora, _ = count(qv, dora=True)
print(f"모델 전체 파라미터: {total:,}")
print(f"LoRA r=8, q·v:      {t_qv:,} ({t_qv / total:.3%})  공식 {lora_formula(qv):,}")
print(f"LoRA r=8, 일곱 모듈: {t_all:,} ({t_all / total:.3%})  공식 {lora_formula(all7):,}")
print(f"DoRA r=8, q·v:      {t_dora:,}  LoRA보다 {t_dora - t_qv:,}개 많다 (크기 벡터 (2048 + 1024) x {L}층 = {(2048 + 1024) * L:,})")
check("LoRA q·v 공식과 실제", t_qv, lora_formula(qv), tol=0)

### 3-1. 학습 메모리 어림 - 3주차의 16바이트 규칙

3주차 규칙: 학습하는 파라미터는 가중치 4 + 그래디언트 4 + Adam 기록 8 = **16바이트**, 얼린 파라미터는 가중치만 4바이트(32비트)다. QLoRA는 얼린 선형 층을 4비트(0.5바이트)로 올린다.

여기서 하나 더 본다. QLoRA가 4비트로 바꾸는 것은 보통 **선형 층**이다. 이 모델은 어휘가 151,936개라 임베딩이 크고, 임베딩은 4비트로 바꾸지 않는 경우가 많다. 아래는 임베딩을 16비트(2바이트)로 두는 경우로 어림한다. 활성값과 양자화 보조 숫자는 빠진 어림이다.

**예측해 보기.** 전체 미세조정, LoRA(q·v, r=8), QLoRA(같은 어댑터) 중 어느 것이 몇 GiB쯤일까?

In [ ]:
emb = cfg.vocab_size * cfg.hidden_size              # 입력 임베딩 (출력 층과 공유)
linear = L * sum(a * b for a, b in shape.values())  # 28층의 선형 층 가중치
rest = total - emb - linear                         # 정규화 층 등
GiB = 2 ** 30
full = total * 16
lora = total * 4 + t_qv * 16
qlora = linear * 0.5 + (emb + rest) * 2 + t_qv * 16
print(f"임베딩 {emb:,} ({emb / total:.1%}), 선형 층 {linear:,} ({linear / total:.1%}), 나머지 {rest:,}")
print(f"전체 미세조정  {full / GiB:5.2f} GiB  (모두 16바이트)")
print(f"LoRA          {lora / GiB:5.2f} GiB  (얼린 가중치 4바이트 + 어댑터 16바이트)")
print(f"QLoRA         {qlora / GiB:5.2f} GiB  (선형 층 0.5바이트, 임베딩 등 2바이트, 어댑터 16바이트)")
print(f"QLoRA에서 임베딩이 차지하는 몫: {(emb * 2) / qlora:.0%}")

## 4. 라벨 확률 (4주차)

프롬프트 다음 토큰 자리에서 세 라벨 첫 토큰의 로짓이 주차 2.0, 시설 1.0, 요금 0.0이다.

**예측해 보기.** (1) 소프트맥스 뒤 세 확률은? (2) 주차와 시설의 확률 비는? (3) 로짓에 모두 5를 더하면 확률은 어떻게 되는가?

In [ ]:
logits = torch.tensor([2.0, 1.0, 0.0])
p = torch.softmax(logits, dim=0)
print("확률 (주차, 시설, 요금):", [round(x, 4) for x in p.tolist()])
print(f"주차 / 시설 = {p[0] / p[1]:.4f} (e^1 = {math.e:.4f})")
print("로짓 + 5:", [round(x, 4) for x in torch.softmax(logits + 5, dim=0).tolist()], "-> 그대로 (차이만 중요하다)")

## 5. 지표와 pass@k (5주차)

긴급 문의 분류 결과: 실제 긴급 10건 중 8건을 긴급으로 잡았고, 실제 일반 30건 중 6건을 긴급으로 잘못 올렸다.

**예측해 보기.** (1) 긴급 부류의 정밀도, 재현율, F1은? (2) 정확도는?

같은 문제를 n = 10번 풀어 c = 3번 맞혔다. **(3)** pass@1과 pass@5의 추정값은? 추정식은 1 - C(n - c, k) / C(n, k)다.

In [ ]:
tp, fn, fp, tn = 8, 2, 6, 24
precision = tp / (tp + fp); recall = tp / (tp + fn)
f1 = 2 * precision * recall / (precision + recall)
acc = (tp + tn) / (tp + fn + fp + tn)
print(f"정밀도 {precision:.4f}, 재현율 {recall:.4f}, F1 {f1:.4f}, 정확도 {acc:.4f}")

def pass_at_k(n, c, k):
    return 1.0 if n - c < k else 1 - math.comb(n - c, k) / math.comb(n, k)
for k in [1, 5]:
    print(f"pass@{k} = {pass_at_k(10, 3, k):.4f}")
check("pass@1 = c / n", pass_at_k(10, 3, 1), 0.3)

## 6. 코사인 유사도와 시각 토큰 (6주차)

**예측해 보기.** (1) (3, 4)와 (4, 3)의 코사인 유사도는? (2) 두 라벨의 코사인이 0.266, 0.274이고 logit_scale이 100이면 두 라벨의 확률 비는? (3) SmolVLM에서 640x640 사진은 분할을 켜면 몇 토큰인가? (긴 변을 2048로 키우고 512 타일로 자른 뒤 전체 사진 1장을 더한다. 장당 64토큰)

In [ ]:
a, b = torch.tensor([3.0, 4.0]), torch.tensor([4.0, 3.0])
cos = torch.nn.functional.cosine_similarity(a, b, dim=0).item()
check("cos((3,4),(4,3))", cos, 0.96)
ratio = math.exp(100 * (0.274 - 0.266))
print(f"확률 비 e^(100 x 0.008) = {ratio:.4f}")
tiles = (2048 // 512) * (2048 // 512) + 1
print(f"640x640 -> 2048x2048, 타일 {tiles - 1}장 + 전체 1장 = {tiles}장 x 64 = {tiles * 64:,}토큰")

## 7. KV 캐시와 RoPE (7주차)

설정: 층 28, KV 헤드 8, 헤드 차원 128, bf16(2바이트), RoPE 기저 1,000,000, 최대 위치 32,768.

**예측해 보기.** (1) 토큰당 KV 캐시는 몇 바이트인가? 8,192토큰이면? (2) KV 헤드가 16개라면 몇 배인가? (3) 가장 빠른 차원 쌍(k = 1)의 주기는? (4) 위치 보간으로 131,072까지 늘리면 위치 65,536은 몇이 되는가?

In [ ]:
per_tok = 2 * 28 * 8 * 128 * 2
print(f"토큰당 {per_tok:,}바이트 = {per_tok / 1024:.0f} KiB, 8,192토큰 {per_tok * 8192 / 2**20:,.0f} MiB")
print(f"KV 헤드 16개면 {2 * 28 * 16 * 128 * 2 / per_tok:.0f}배")
b, d = 1_000_000, 128
print(f"k = 1 주기: 2π x b^0 = {2 * math.pi:.2f}토큰, k = 64 주기: {2 * math.pi * b ** ((d - 2) / d):,.0f}토큰")
print(f"위치 보간: 65,536 x 32,768 / 131,072 = {65536 * 32768 // 131072:,}")

## 8. 돌아보기

1. 틀린 문항은 몇 번이었나요? 각 문항이 어느 주차의 어떤 개념에서 왔는지 적어 보세요.
2. 3에서 LoRA를 일곱 모듈 모두에 붙이면 q·v에만 붙일 때보다 몇 배가 되었나요? 그래도 전체의 몇 %인가요?
3. 3-1에서 QLoRA의 메모리 중 임베딩이 차지하는 몫은 얼마였나요? 어휘가 큰 모델에서 QLoRA의 이득이 3주차 표보다 작아지는 이유를 한 문장으로 써 보세요.
4. 1~7주차 개념 위계에서 세 문장으로 설명하지 못한 마디는 어디였나요?

답은 아래 셀에 글로 적으면 됩니다.

*(여기에 답을 적으세요)*

## 9. 제출

1. 상단 메뉴 **파일 > .ipynb 다운로드** 로 이 노트북을 내려받습니다
2. [저장소](https://github.com/halla-ai/deepnlp-2026)의 `assignments/week-08/<내 학번>/` 에 업로드합니다
3. Pull Request를 엽니다

자세한 방법은 강의 사이트의 **과제 제출** 문서에 있습니다.

---

**막혔나요?** 오류 메시지의 마지막 줄을 먼저 읽어 보세요. 그래도 안 되면 AI Professor 튜터에게 묻고, 그래도 막히면 저장소 Issues에 남기세요.